In [1]:
# 第3章｜作業：哪些財報數據重要：每次開啟 notebook，先執行這一格（安裝套件、登入 FinLab、開啟資料快取）
%pip install -q finlab==2.0.21 ta-lib==0.8.1 lightgbm
import finlab
from finlab import data

finlab.login()
data.set_storage(data.FileStorage())

Note: you may need to restart the kernel to use updated packages.


已登入（使用快取憑證）。


In [2]:
QUICK_RUN = False  # True：減少重複次數，只用來快速檢查程式能不能跑

# 作業：哪些財報數據重要？

**對應影片**：第 3 章 單元 11「作業：哪些財報數據重要（feature importance）」

**和影片的差異**：資料集改用 `finlab.ml.feature` 與 `finlab.ml.label` 建立；除了模型內建的重要性，
另外介紹更可靠的 **permutation importance**（打亂某個特徵後，模型在測試資料上變差多少）。

這本是**起始 notebook**：從頭執行會完成三種重要性的計算與比較。你的任務寫在最後的「作業」段落。

In [3]:
import numpy as np
import pandas as pd
import finlab.ml.feature as feature
import finlab.ml.label as label
from lightgbm import LGBMClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score

REBALANCE = 'QE'
TRAIN_END = '2016-12-31'
RANDOM_STATE = 0
MIN_FEATURE_COVERAGE = 0.5

## 1. 資料集（和隨機森林單元相同）

In [4]:
feature_names = data.search('fundamental_features').str.split(':').str[1].tolist()

dataset = feature.combine(
    {name: data.get(f'fundamental_features:{name}') for name in feature_names}, resample=REBALANCE,
).replace([np.inf, -np.inf], np.nan)
dataset['beat_median'] = label.excess_over_median(dataset.index, resample=REBALANCE) > 0
dataset['return'] = label.return_percentage(dataset.index, resample=REBALANCE)
dataset = dataset.dropna(subset=['return'])
dataset = dataset[dataset[feature_names].notna().mean(axis=1) >= MIN_FEATURE_COVERAGE]

dates = dataset.index.get_level_values('datetime')
train, test = dataset[dates <= TRAIN_END], dataset[dates > TRAIN_END]
print(f'訓練資料 {len(train):,} 筆、測試資料 {len(test):,} 筆、{len(feature_names)} 個特徵')

訓練資料 22,537 筆、測試資料 11,475 筆、53 個特徵


## 2. 訓練 LightGBM

In [5]:
def train_model(columns: list[str]) -> LGBMClassifier:
    model = LGBMClassifier(
        n_estimators=300, learning_rate=0.03, num_leaves=15, min_child_samples=200,
        subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=RANDOM_STATE, verbose=-1,
    )
    return model.fit(train[columns], train['beat_median'])


def test_auc(model: LGBMClassifier, columns: list[str]) -> float:
    return roc_auc_score(test['beat_median'], model.predict_proba(test[columns])[:, 1])


model = train_model(feature_names)
print(f'測試 AUC：{test_auc(model, feature_names):.3f}')

測試 AUC：0.574


## 3. 三種重要性

| 名稱 | 算法 | 缺點 |
| --- | --- | --- |
| split | 這個特徵被拿來分裂幾次 | 數值種類多的特徵容易被高估 |
| gain | 這個特徵的分裂總共降低多少 loss | 只反映訓練資料，過度擬合的特徵也會很高 |
| permutation | 在**測試資料**上把這個特徵隨機打亂，AUC 下降多少 | 計算比較久；高度相關的特徵會互相分掉重要性 |

In [6]:
N_REPEATS = 2 if QUICK_RUN else 5
SAMPLE_FRACTION = 0.5  # 每次只打亂一半的測試資料，計算量約為全量的四分之一

permutation = permutation_importance(
    model, test[feature_names], test['beat_median'], scoring='roc_auc',
    n_repeats=N_REPEATS, max_samples=SAMPLE_FRACTION, random_state=RANDOM_STATE, n_jobs=1,
)

importance = pd.DataFrame({
    'split': model.booster_.feature_importance(importance_type='split'),
    'gain': model.booster_.feature_importance(importance_type='gain'),
    'permutation': permutation.importances_mean,
}, index=feature_names)
importance = importance / importance.abs().sum()
importance.sort_values('permutation', ascending=False).head(20).style.bar(color='#f4a261', align='zero').format('{:.3f}')

,split,gain,permutation
淨值成長率,0.042,0.045,0.073
稅前淨利成長率,0.021,0.022,0.053
營業利益,0.013,0.033,0.046
營收成長率,0.032,0.030,0.045
研究發展費用率,0.036,0.038,0.043
營業利益成長率,0.025,0.023,0.042
ROE綜合損益,0.022,0.019,0.040
經常利益成長率,0.017,0.018,0.038
貝里比率,0.023,0.049,0.037
EBITDA,0.010,0.012,0.031


三種方法排名的相關程度（Spearman 等級相關）：

In [7]:
importance.corr(method='spearman').round(2)

,split,gain,permutation
split,1.00,0.92,0.20
gain,0.92,1.00,0.32
permutation,0.20,0.32,1.00


## 4. 只用重要的特徵，模型會不會更好？

依 permutation importance 排序，只保留前 k 個特徵重新訓練。
注意：這裡為了示範直接用測試資料挑特徵，嚴格來說應該再切一段驗證資料來挑（作業第 3 題）。

In [8]:
TOP_K_CHOICES = [5, 10, 20, len(feature_names)]

ranked = importance['permutation'].sort_values(ascending=False).index.tolist()
pd.Series({k: test_auc(train_model(ranked[:k]), ranked[:k]) for k in TOP_K_CHOICES},
          name='test AUC').rename_axis('top k features').to_frame().style.format('{:.3f}')

,test AUC
top k features,
5,0.575
10,0.575
20,0.572
53,0.570


## 作業

1. **比較模型**：用隨機森林（`RandomForestClassifier`）重算 permutation importance，前 10 名和 LightGBM 一樣嗎？
2. **看方向**：挑 permutation importance 前 3 名的特徵，每季依該特徵分成 5 組，比較各組的平均 `return`。是越大越好、越小越好，還是中間最好？
3. **避免偷看**：把 2016 年切出來當驗證資料，只用驗證資料挑特徵數量 k，最後才在 2017 年之後的測試資料上評估一次。
4. **穩定性**：分別用 2013–2014、2015–2016 的資料訓練，重要特徵的排名穩定嗎？不穩定的特徵，你還會相信它嗎？